In [1]:
# cpr_srd_vol_ice_detection
from osgeo import gdal
import numpy as np
import os
from pathlib import Path

gdal.PushErrorHandler('CPLQuietErrorHandler')

base_dir = r"E:\ISRO_IMG"

# Find all south pole folders (sp)
sp_folders = []
for folder in os.listdir(base_dir):
    if 'mpcpsp' in folder or 'my4rsp' in folder:
        sp_folders.append(os.path.join(base_dir, folder))

print(f"Found {len(sp_folders)} south pole folders\n")

# Process each folder
for folder_path in sorted(sp_folders):
    folder_name = os.path.basename(folder_path)
    print(f"{'='*60}")
    print(f"Processing: {folder_name}")
    print(f"{'='*60}")
    
    # Find all TIF files in this folder
    data_dir = os.path.join(folder_path, 'data', 'derived')
    
    if not os.path.exists(data_dir):
        print(f"Data dir not found: {data_dir}\n")
        continue
    
    # Get all TIF files
    tif_files = {}
    for root, dirs, files in os.walk(data_dir):
        for f in files:
            if f.endswith('.tif'):
                full_path = os.path.join(root, f)
                # Extract product type from filename
                if '_d_cpr_' in f:
                    tif_files['cpr'] = full_path
                elif '_d_srd_' in f:
                    tif_files['srd'] = full_path
                elif '_d_vol_' in f:
                    tif_files['vol'] = full_path
                elif '_d_odd_' in f:
                    tif_files['odd'] = full_path
                elif '_d_evn_' in f:
                    tif_files['evn'] = full_path
    
    # Load and analyze each product
    data = {}
    for product_name, file_path in tif_files.items():
        try:
            ds = gdal.Open(file_path)
            band = ds.GetRasterBand(1)
            array = band.ReadAsArray().astype(np.float32)
            
            # Get statistics (excluding NaN)
            valid = array[~np.isnan(array)]
            
            data[product_name] = {
                'array': array,
                'min': np.nanmin(array),
                'max': np.nanmax(array),
                'mean': np.nanmean(array),
                'median': np.nanmedian(array),
                'valid_count': len(valid)
            }
            
            print(f"\n{product_name.upper()}:")
            print(f"  Range:  {data[product_name]['min']:.6f} → {data[product_name]['max']:.6f}")
            print(f"  Mean:   {data[product_name]['mean']:.6f}")
            print(f"  Median: {data[product_name]['median']:.6f}")
            print(f"  Valid pixels: {data[product_name]['valid_count']:,}")
            
        except Exception as e:
            print(f"ERROR loading {product_name}: {e}")
    
    # Compute ice detection using different metrics
    print(f"\n{'─'*60}")
    print("Ice Detection Results:")
    print(f"{'─'*60}")
    
    # Method 1: Using CPR + SRD
    if 'cpr' in data and 'srd' in data:
        cpr = data['cpr']['array']
        srd = data['srd']['array']
        ice_cpr_srd = (cpr > 1.0) & (srd < 0.2)
        print(f"CPR>1.0 + SRD<0.2: {ice_cpr_srd.sum():,} pixels")
    
    # Method 2: Using VOL (volume scatter)
    if 'vol' in data:
        vol = data['vol']['array']
        # High volume scatter = ice
        # Threshold: top 10% of valid pixels
        vol_valid = vol[~np.isnan(vol)]
        vol_threshold = np.percentile(vol_valid, 90)
        ice_vol = vol > vol_threshold
        print(f"VOL > {vol_threshold:.4f} (90th percentile): {ice_vol.sum():,} pixels")
        
        # Also show lower thresholds
        vol_75 = np.percentile(vol_valid, 75)
        ice_vol_75 = vol > vol_75
        print(f"VOL > {vol_75:.4f} (75th percentile): {ice_vol_75.sum():,} pixels")
    
    # Method 3: Using ODD bounce (volume scattering proxy)
    if 'odd' in data:
        odd = data['odd']['array']
        odd_valid = odd[~np.isnan(odd)]
        odd_threshold = np.percentile(odd_valid, 90)
        ice_odd = odd > odd_threshold
        print(f"ODD > {odd_threshold:.4f} (90th percentile): {ice_odd.sum():,} pixels")
    
    print()

print(f"\n{'='*60}")
print("Summary: All folders processed")
print(f"{'='*60}")

Found 4 south pole folders

Processing: ch2_sar_ndxl_20250630mpcpspeast_d_fp_xxx


c:\Users\KIIT\anaconda3\envs\moon_ice_mapping\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(



CPR:
  Range:  0.002990 → 4.078197
  Mean:   0.281229
  Median: 0.248386
  Valid pixels: 54,912,646

SRD:
  Range:  0.001845 → 0.999578
  Mean:   0.794704
  Median: 0.809044
  Valid pixels: 54,905,196

────────────────────────────────────────────────────────────
Ice Detection Results:
────────────────────────────────────────────────────────────
CPR>1.0 + SRD<0.2: 2,787 pixels

Processing: ch2_sar_ndxl_20250630mpcpspwest_d_fp_xxx

CPR:
  Range:  0.000888 → 4.919318
  Mean:   0.253207
  Median: 0.222350
  Valid pixels: 340,343,477

SRD:
  Range:  0.000630 → 20.951334
  Mean:   0.813398
  Median: 0.826301
  Valid pixels: 340,317,714

────────────────────────────────────────────────────────────
Ice Detection Results:
────────────────────────────────────────────────────────────
CPR>1.0 + SRD<0.2: 3,814 pixels

Processing: ch2_sar_ndxl_20250630my4rspeast_d_fp_xxx

EVN:
  Range:  0.000000 → 0.000000
  Mean:   0.000000
  Median: 0.000000
  Valid pixels: 54,840,693

ODD:
  Range:  0.000000 → 0

In [2]:
from osgeo import gdal, osr
import numpy as np
import os
from pathlib import Path

gdal.PushErrorHandler('CPLQuietErrorHandler')

# Base directory
base_dir = r"E:\ISRO_IMG"
output_dir = r"E:\output\ice_maps"
os.makedirs(output_dir, exist_ok=True)

print("="*70)
print("BATCH ICE DETECTION & VOLUME ESTIMATION")
print("="*70)

# Find all south pole CPR files
cpr_files = []
for root, dirs, files in os.walk(base_dir):
    for f in files:
        if '_d_cpr_' in f and f.endswith('.tif'):
            # Only south pole
            if 'mpcpsp' in f or 'my4rsp' in f:
                cpr_files.append(os.path.join(root, f))

cpr_files.sort()
print(f"\nFound {len(cpr_files)} south pole CPR files\n")

# Process each
summary_results = []

for cpr_path in cpr_files:
    # Find matching SRD file
    srd_path = cpr_path.replace('_d_cpr_', '_d_srd_')
    
    if not os.path.exists(srd_path):
        print(f"⚠ WARNING: No matching SRD for {os.path.basename(cpr_path)}")
        continue
    
    folder_name = os.path.basename(os.path.dirname(cpr_path))
    print(f"Processing: {folder_name}")
    print("-" * 70)
    
    # Load data
    cpr_ds = gdal.Open(cpr_path)
    srd_ds = gdal.Open(srd_path)
    
    if cpr_ds is None or srd_ds is None:
        print(f"  ERROR: Could not open files\n")
        continue
    
    try:
        cpr = cpr_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
        srd = srd_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
        
        crs_wkt = cpr_ds.GetProjection()
        geotransform = cpr_ds.GetGeoTransform()
        
        # Compute ice probability
        ice_probability = np.zeros_like(cpr)
        
        # High confidence: CPR > 1.0 AND SRD < 0.2
        high_conf = (cpr > 1.0) & (srd < 0.2)
        ice_probability[high_conf] = 1.0
        
        # Medium confidence: CPR > 0.8 AND SRD < 0.3
        med_conf = (cpr > 0.8) & (srd < 0.3) & ~high_conf
        ice_probability[med_conf] = 0.5
        
        ice_high = high_conf.sum()
        ice_med = med_conf.sum()
        
        print(f"  High conf ice pixels: {ice_high:,}")
        print(f"  Med conf ice pixels: {ice_med:,}")
        
        # Save ice probability map
        output_name = os.path.basename(cpr_path).replace('_d_cpr_', '_ice_prob_')
        output_path = os.path.join(output_dir, output_name)
        
        driver = gdal.GetDriverByName('GTiff')
        out_ds = driver.Create(output_path, cpr.shape[1], cpr.shape[0], 
                              1, gdal.GDT_Float32)
        out_ds.SetGeoTransform(geotransform)
        out_ds.SetProjection(crs_wkt)
        out_ds.GetRasterBand(1).WriteArray(ice_probability)
        out_ds.FlushCache()
        del out_ds
        
        print(f"  ✓ Saved: {output_name}")
        
        # Volume estimation
        pixel_size_m = 30
        pixel_area_m2 = pixel_size_m ** 2
        depth_m = 5.0
        
        ice_pixels_eff = ice_high + 0.5 * ice_med
        regolith_volume_m3 = ice_pixels_eff * pixel_area_m2 * depth_m
        
        conc_min, conc_best, conc_max = 0.02, 0.05, 0.08
        ice_vol_min = regolith_volume_m3 * conc_min
        ice_vol_best = regolith_volume_m3 * conc_best
        ice_vol_max = regolith_volume_m3 * conc_max
        
        print(f"\n  Ice Volume Estimates:")
        print(f"    Conservative (2%):  {ice_vol_min:.3e} m³ ({ice_vol_min/1e6:.2f}M m³)")
        print(f"    Best (5%):          {ice_vol_best:.3e} m³ ({ice_vol_best/1e6:.2f}M m³)")
        print(f"    Optimistic (8%):    {ice_vol_max:.3e} m³ ({ice_vol_max/1e6:.2f}M m³)")
        
        summary_results.append({
            'folder': folder_name,
            'high_conf': ice_high,
            'med_conf': ice_med,
            'regolith_vol_m3': regolith_volume_m3,
            'ice_vol_min': ice_vol_min,
            'ice_vol_best': ice_vol_best,
            'ice_vol_max': ice_vol_max
        })
        
        print()
        
    except Exception as e:
        print(f"  ERROR: {e}\n")
        continue

# Final summary
print("\n" + "="*70)
print("SUMMARY — ALL SOUTH POLE FOLDERS")
print("="*70)

if summary_results:
    total_high = sum(r['high_conf'] for r in summary_results)
    total_med = sum(r['med_conf'] for r in summary_results)
    total_regolith = sum(r['regolith_vol_m3'] for r in summary_results)
    total_ice_min = sum(r['ice_vol_min'] for r in summary_results)
    total_ice_best = sum(r['ice_vol_best'] for r in summary_results)
    total_ice_max = sum(r['ice_vol_max'] for r in summary_results)
    
    print(f"\nDetailed Results:")
    print(f"{'Folder':<35} {'High Conf':>12} {'Med Conf':>12} {'Ice Vol Best':>20}")
    print("-" * 80)
    for r in summary_results:
        print(f"{r['folder']:<35} {r['high_conf']:>12,} {r['med_conf']:>12,} {r['ice_vol_best']:>20.2e}")
    
    print("-" * 80)
    print(f"{'TOTAL':<35} {total_high:>12,} {total_med:>12,} {total_ice_best:>20.2e}")
    
    print(f"\nGlobal South Pole Ice Volume:")
    print(f"  Range: [{total_ice_min:.3e}, {total_ice_max:.3e}] m³")
    print(f"  Best estimate: {total_ice_best:.3e} m³")
    print(f"  In millions: {total_ice_best/1e6:.1f} million m³")
    
    print(f"\nAll ice maps saved to: {output_dir}")
else:
    print("No results processed.")

BATCH ICE DETECTION & VOLUME ESTIMATION

Found 2 south pole CPR files

Processing: 20250630
----------------------------------------------------------------------
  High conf ice pixels: 2,787
  Med conf ice pixels: 13,658
  ✓ Saved: ch2_sar_ndxl_20250630mpcpspeast_ice_prob_xx_fp_xx_xxx.tif

  Ice Volume Estimates:
    Conservative (2%):  8.654e+05 m³ (0.87M m³)
    Best (5%):          2.164e+06 m³ (2.16M m³)
    Optimistic (8%):    3.462e+06 m³ (3.46M m³)

Processing: 20250630
----------------------------------------------------------------------
  High conf ice pixels: 3,814
  Med conf ice pixels: 30,196
  ✓ Saved: ch2_sar_ndxl_20250630mpcpspwest_ice_prob_xx_fp_xx_xxx.tif

  Ice Volume Estimates:
    Conservative (2%):  1.702e+06 m³ (1.70M m³)
    Best (5%):          4.255e+06 m³ (4.26M m³)
    Optimistic (8%):    6.808e+06 m³ (6.81M m³)


SUMMARY — ALL SOUTH POLE FOLDERS

Detailed Results:
Folder                                 High Conf     Med Conf         Ice Vol Best
-----------

In [3]:
# ======================================================================
# VALIDATION AGAINST SINHA ET AL. (2026)
# ======================================================================
from osgeo import gdal
import numpy as np
import matplotlib.pyplot as plt
import os

gdal.PushErrorHandler('CPLQuietErrorHandler')

base_dir = r"E:\ISRO_IMG"

print("="*70)
print("VALIDATION: CPR STATISTICS vs SINHA ET AL. (2026)")
print("="*70)

# Find CPR files
cpr_files = []
for root, dirs, files in os.walk(base_dir):
    for f in files:
        if '_d_cpr_' in f and 'mpcpsp' in f and f.endswith('.tif'):
            cpr_files.append(os.path.join(root, f))

cpr_files.sort()

# Sinha et al. 2026 reference values
reference = {
    'F2 (ice)': {'CPR': 1.95, 'DOP': 0.10},
    'F3 (ice)': {'CPR': 1.80, 'DOP': 0.11},
    'H3 (ice)': {'CPR': 1.57, 'DOP': 0.12},
    'S1 (ice)': {'CPR': 1.40, 'DOP': 0.13},
    'H1 (no ice)': {'CPR': 0.80, 'DOP': 0.45},
}

print("\nSinha et al. (2026) Reference Values:")
print("-" * 70)
for crater, vals in reference.items():
    print(f"{crater:<20} CPR={vals['CPR']:.2f}, DOP={vals['DOP']:.2f}")

print("\n\nYour Data Statistics:")
print("="*70)

all_cpr_data = []
swath_stats = []

for cpr_path in cpr_files:
    folder_name = os.path.basename(os.path.dirname(cpr_path))
    swath_name = os.path.basename(cpr_path).split('_')[6:8]  # Extract swath name
    swath_name = '_'.join(swath_name)
    
    ds = gdal.Open(cpr_path)
    cpr = ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
    
    # Remove NaN
    cpr_valid = cpr[~np.isnan(cpr)]
    all_cpr_data.append(cpr_valid)
    
    print(f"\nSwath: {swath_name}")
    print("-" * 70)
    print(f"  Total pixels: {cpr.size:,}")
    print(f"  Valid pixels: {len(cpr_valid):,}")
    print(f"  NaN pixels: {np.isnan(cpr).sum():,}")
    
    print(f"\n  CPR Statistics:")
    print(f"    Min:        {np.nanmin(cpr):.6f}")
    print(f"    5th %ile:   {np.percentile(cpr_valid, 5):.6f}")
    print(f"    25th %ile:  {np.percentile(cpr_valid, 25):.6f}")
    print(f"    Median:     {np.percentile(cpr_valid, 50):.6f}")
    print(f"    Mean:       {np.nanmean(cpr):.6f}")
    print(f"    75th %ile:  {np.percentile(cpr_valid, 75):.6f}")
    print(f"    95th %ile:  {np.percentile(cpr_valid, 95):.6f}")
    print(f"    Max:        {np.nanmax(cpr):.6f}")
    print(f"    Std Dev:    {np.nanstd(cpr):.6f}")
    
    # Ice detection thresholds
    print(f"\n  Ice Detection Breakdown:")
    ice_count_05 = (cpr >= 0.5).sum()
    ice_count_08 = (cpr >= 0.8).sum()
    ice_count_10 = (cpr >= 1.0).sum()
    ice_count_15 = (cpr >= 1.5).sum()
    ice_count_20 = (cpr >= 2.0).sum()
    
    print(f"    CPR >= 0.5: {ice_count_05:,} pixels ({100*ice_count_05/len(cpr_valid):.2f}%)")
    print(f"    CPR >= 0.8: {ice_count_08:,} pixels ({100*ice_count_08/len(cpr_valid):.2f}%)")
    print(f"    CPR >= 1.0: {ice_count_10:,} pixels ({100*ice_count_10/len(cpr_valid):.2f}%) ← Ice threshold")
    print(f"    CPR >= 1.5: {ice_count_15:,} pixels ({100*ice_count_15/len(cpr_valid):.2f}%)")
    print(f"    CPR >= 2.0: {ice_count_20:,} pixels ({100*ice_count_20/len(cpr_valid):.2f}%) ← Strong ice")
    
    swath_stats.append({
        'swath': swath_name,
        'cpr_mean': np.nanmean(cpr),
        'cpr_median': np.percentile(cpr_valid, 50),
        'cpr_max': np.nanmax(cpr),
        'ice_pixels': ice_count_10,
        'valid_pixels': len(cpr_valid)
    })

# Combined statistics
print("\n\n" + "="*70)
print("COMBINED SOUTH POLE STATISTICS")
print("="*70)

all_cpr = np.concatenate(all_cpr_data)
print(f"\nGlobal South Pole CPR Distribution:")
print(f"  Min:        {np.min(all_cpr):.6f}")
print(f"  25th %ile:  {np.percentile(all_cpr, 25):.6f}")
print(f"  Median:     {np.percentile(all_cpr, 50):.6f}")
print(f"  Mean:       {np.mean(all_cpr):.6f}")
print(f"  75th %ile:  {np.percentile(all_cpr, 75):.6f}")
print(f"  95th %ile:  {np.percentile(all_cpr, 95):.6f}")
print(f"  Max:        {np.max(all_cpr):.6f}")

print(f"\nIce Pixels (CPR >= 1.0): {(all_cpr >= 1.0).sum():,}")
print(f"Strong Ice (CPR >= 1.5): {(all_cpr >= 1.5).sum():,}")
print(f"Very Strong (CPR >= 2.0): {(all_cpr >= 2.0).sum():,}")

# Comparison with literature
print("\n" + "="*70)
print("VALIDATION AGAINST SINHA ET AL. (2026)")
print("="*70)

print(f"\nYour max CPR: {np.max(all_cpr):.3f}")
print(f"Sinha F2 (strongest ice signal): {reference['F2 (ice)']['CPR']:.3f}")

if np.max(all_cpr) >= 1.5:
    print("✓ Your data captures strong ice signatures (CPR > 1.5)")
elif np.max(all_cpr) >= 1.0:
    print("✓ Your data captures moderate ice signatures (CPR > 1.0)")
else:
    print("✗ WARNING: Your CPR max is below expected ice threshold")

print(f"\nYour median CPR: {np.median(all_cpr):.3f}")
print(f"Expected background (H1, no ice): {reference['H1 (no ice)']['CPR']:.3f}")

if np.median(all_cpr) < 0.5:
    print("✓ Background CPR is low (mostly non-ice)")
else:
    print("⚠ Background CPR is higher than expected")

# Create histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CPR histogram
axes[0].hist(all_cpr, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Ice threshold (CPR=1.0)')
axes[0].axvline(np.median(all_cpr), color='green', linestyle='--', linewidth=2, label=f'Median ({np.median(all_cpr):.3f})')
axes[0].axvline(reference['F2 (ice)']['CPR'], color='orange', linestyle='--', linewidth=2, label=f"F2 reference ({reference['F2 (ice)']['CPR']:.2f})")
axes[0].set_xlabel('CPR (Circular Polarization Ratio)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('CPR Distribution (South Pole)', fontsize=12)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_cpr = np.sort(all_cpr)
cumulative = np.arange(1, len(sorted_cpr) + 1) / len(sorted_cpr)
axes[1].plot(sorted_cpr, cumulative, linewidth=2, color='steelblue')
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Ice threshold')
axes[1].axvline(reference['F2 (ice)']['CPR'], color='orange', linestyle='--', linewidth=2, label='F2 reference')
axes[1].set_xlabel('CPR', fontsize=12)
axes[1].set_ylabel('Cumulative Fraction', fontsize=12)
axes[1].set_title('Cumulative CPR Distribution', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r'E:\output\cpr_validation.png', dpi=150, bbox_inches='tight')
print(f"\n✓ Saved: E:\output\cpr_validation.png")
plt.close()

print("\n" + "="*70)
print("VALIDATION COMPLETE")
print("="*70)

<>:169: SyntaxWarning: invalid escape sequence '\o'
<>:169: SyntaxWarning: invalid escape sequence '\o'
C:\Users\KIIT\AppData\Local\Temp\ipykernel_24652\1082533010.py:169: SyntaxWarning: invalid escape sequence '\o'
  print(f"\n✓ Saved: E:\output\cpr_validation.png")


VALIDATION: CPR STATISTICS vs SINHA ET AL. (2026)

Sinha et al. (2026) Reference Values:
----------------------------------------------------------------------
F2 (ice)             CPR=1.95, DOP=0.10
F3 (ice)             CPR=1.80, DOP=0.11
H3 (ice)             CPR=1.57, DOP=0.12
S1 (ice)             CPR=1.40, DOP=0.13
H1 (no ice)          CPR=0.80, DOP=0.45


Your Data Statistics:

Swath: xx_fp
----------------------------------------------------------------------
  Total pixels: 156,560,178
  Valid pixels: 54,912,646
  NaN pixels: 101,647,532

  CPR Statistics:
    Min:        0.002990
    5th %ile:   0.086283
    25th %ile:  0.173580
    Median:     0.248386
    Mean:       0.281229
    75th %ile:  0.350660
    95th %ile:  0.587341
    Max:        4.078197
    Std Dev:    0.161317

  Ice Detection Breakdown:
    CPR >= 0.5: 4,902,943 pixels (8.93%)
    CPR >= 0.8: 708,257 pixels (1.29%)
    CPR >= 1.0: 203,924 pixels (0.37%) ← Ice threshold
    CPR >= 1.5: 6,912 pixels (0.01%)
    CP

C:\Users\KIIT\AppData\Local\Temp\ipykernel_24652\1082533010.py:167: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\KIIT\AppData\Local\Temp\ipykernel_24652\1082533010.py:168: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(r'E:\output\cpr_validation.png', dpi=150, bbox_inches='tight')



✓ Saved: E:\output\cpr_validation.png

VALIDATION COMPLETE


In [4]:
#  To check the projection
from osgeo import gdal

cpr_east = r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspeast_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspeast_d_cpr_xx_fp_xx_xxx.tif"
cpr_west = r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspwest_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspwest_d_cpr_xx_fp_xx_xxx.tif"

for label, path in [("EAST", cpr_east), ("WEST", cpr_west)]:
    ds = gdal.Open(path)
    print(f"\n{label}:")
    print(f"Projection: {ds.GetProjection()[:100]}...")  # First 100 chars
    print(f"Geotransform: {ds.GetGeoTransform()}")
    print(f"Shape: {ds.RasterXSize} x {ds.RasterYSize}")


EAST:
Projection: PROJCS["Moon_2000_South_Pole_Stereographic",GEOGCS["GCS_Moon_2000",DATUM["D_Moon_2000",SPHEROID["Moo...
Geotransform: (-159351.0785, 25.0, 0.0, 149109.7303, 0.0, -25.0)
Shape: 12794 x 12237

WEST:
Projection: PROJCS["Moon_2000_South_Pole_Stereographic",GEOGCS["GCS_Moon_2000",DATUM["D_Moon_2000",SPHEROID["Moo...
Geotransform: (-305557.4188, 25.0, 0.0, 315421.2535, 0.0, -25.0)
Shape: 24181 x 24794


In [6]:
# Per crater ice detection using corrected coordinates
from osgeo import gdal, osr
import numpy as np

gdal.PushErrorHandler('CPLQuietErrorHandler')

# Files
files = {
    'east': {
        'cpr': r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspeast_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspeast_d_cpr_xx_fp_xx_xxx.tif",
        'srd': r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspeast_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspeast_d_srd_xx_fp_xx_xxx.tif",
    },
    'west': {
        'cpr': r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspwest_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspwest_d_cpr_xx_fp_xx_xxx.tif",
        'srd': r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspwest_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspwest_d_srd_xx_fp_xx_xxx.tif",
    }
}

# Craters
craters = {
    'F2_Faustini': {'lat': -87.39, 'lon': 82.31, 'expected_cpr': 1.95},
    'H3_Haworth': {'lat': -86.09, 'lon': 97.50, 'expected_cpr': 1.57},
    'H1_Haworth': {'lat': -86.09, 'lon': 98.00, 'expected_cpr': 0.80},
    'S1_Shoemaker': {'lat': -89.33, 'lon': 76.92, 'expected_cpr': 1.40},
}

# Load swaths
swath_data = {}
for swath_name, swath_files in files.items():
    cpr_ds = gdal.Open(swath_files['cpr'])
    srd_ds = gdal.Open(swath_files['srd'])
    
    swath_data[swath_name] = {
        'cpr': cpr_ds.GetRasterBand(1).ReadAsArray().astype(np.float32),
        'srd': srd_ds.GetRasterBand(1).ReadAsArray().astype(np.float32),
        'geotransform': cpr_ds.GetGeoTransform(),
        'projection_wkt': cpr_ds.GetProjection(),
        'shape': (cpr_ds.RasterYSize, cpr_ds.RasterXSize),
        'ds': cpr_ds
    }
    print(f"Loaded {swath_name.upper()}: {swath_data[swath_name]['shape']}")

def latlon_to_pixel_correct(lat, lon, ds, radius_m=2000):
    """
    Convert Moon lat/lon to pixel coordinates using GDAL's native transformation.
    """
    # Create source (geographic Moon) SRS
    source_srs = osr.SpatialReference()
    source_srs.ImportFromWkt("""GEOGCS["GCS_Moon_2000",
        DATUM["D_Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],
        PRIMEM["Reference_Meridian",0],
        UNIT["degree",0.0174532925199433]]""")
    
    # Target SRS from dataset
    target_srs = osr.SpatialReference()
    target_srs.ImportFromWkt(ds.GetProjection())
    
    # Create transformation
    transform = osr.CoordinateTransformation(source_srs, target_srs)
    x, y, z = transform.TransformPoint(lon, lat)
    
    # Get geotransform
    gt = ds.GetGeoTransform()
    
    # Convert to pixel coordinates
    pixel_col = (x - gt[0]) / gt[1]
    pixel_row = (y - gt[3]) / gt[5]
    
    return int(pixel_row), int(pixel_col), x, y

print("\n" + "="*70)
print("PER-CRATER ICE DETECTION (CORRECTED COORDINATES)")
print("="*70)

crater_results = []

for crater_name, crater_info in craters.items():
    lat = crater_info['lat']
    lon = crater_info['lon']
    expected_cpr = crater_info['expected_cpr']
    radius_m = 2000
    
    print(f"\n{crater_name}:")
    print(f"  Location: {lat}°S, {lon}°E")
    print(f"  Expected CPR: {expected_cpr:.2f}")
    
    found = False
    for swath_name, swath in swath_data.items():
        try:
            row, col, x, y = latlon_to_pixel_correct(lat, lon, swath['ds'], radius_m)
            
            # Check bounds
            if 0 <= row < swath['shape'][0] and 0 <= col < swath['shape'][1]:
                radius_pixels = radius_m / 25.0
                
                # Extract circular region
                yy, xx = np.ogrid[:swath['shape'][0], :swath['shape'][1]]
                mask = (xx - col)**2 + (yy - row)**2 <= radius_pixels**2
                
                cpr_region = swath['cpr'][mask]
                srd_region = swath['srd'][mask]
                
                # Create combined valid mask (both CPR and SRD must be valid)
                valid_mask = ~np.isnan(cpr_region) & ~np.isnan(srd_region)
                cpr_valid = cpr_region[valid_mask]
                srd_valid = srd_region[valid_mask]
                
                if len(cpr_valid) > 100:  # Need enough pixels
                    cpr_mean = np.mean(cpr_valid)
                    cpr_max = np.max(cpr_valid)
                    cpr_median = np.median(cpr_valid)
                    srd_mean = np.mean(srd_valid)
                    
                    print(f"\n  ✓ Found in {swath_name.upper()}:")
                    print(f"    Pixel: row={row}, col={col}")
                    print(f"    Stereographic: X={x:.0f}m, Y={y:.0f}m")
                    print(f"    Pixels: {len(cpr_valid):,}")
                    
                    print(f"\n    CPR:")
                    print(f"      Mean:   {cpr_mean:.4f}")
                    print(f"      Median: {cpr_median:.4f}")
                    print(f"      Max:    {cpr_max:.4f}")
                    print(f"      Expected: {expected_cpr:.4f}")
                    
                    error_pct = abs(cpr_mean - expected_cpr) / expected_cpr * 100
                    if error_pct < 30:
                        print(f"      ✓ MATCH ({error_pct:.1f}% error)")
                    else:
                        print(f"      ⚠ {error_pct:.1f}% error from expected")
                    
                    print(f"\n    SRD: mean={srd_mean:.4f}")
                    
                    ice_high = ((cpr_valid > 1.0) & (srd_valid < 0.2)).sum()
                    ice_med = ((cpr_valid > 0.8) & (srd_valid < 0.3)).sum()
                    print(f"    Ice (CPR>1.0, SRD<0.2): {ice_high:,} pixels")
                    print(f"    Ice (CPR>0.8, SRD<0.3): {ice_med:,} pixels")
                    
                    crater_results.append({
                        'crater': crater_name,
                        'swath': swath_name,
                        'cpr_mean': cpr_mean,
                        'expected': expected_cpr,
                        'error_pct': error_pct,
                        'ice_high': ice_high,
                        'pixels': len(cpr_valid)
                    })
                    
                    found = True
                    break
        except Exception as e:
            print(f"    Error in {swath_name}: {e}")

    if not found:
        print(f"  ⚠ Not found")

# Summary
if crater_results:
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"\n{'Crater':<20} {'CPR':<10} {'Expected':<10} {'Error':<10} {'Ice Pixels':<10}")
    print("-"*70)
    for r in crater_results:
        print(f"{r['crater']:<20} {r['cpr_mean']:<10.4f} {r['expected']:<10.4f} {r['error_pct']:<10.1f}% {r['ice_high']:<10,}")


Loaded EAST: (12237, 12794)
Loaded WEST: (24794, 24181)

PER-CRATER ICE DETECTION (CORRECTED COORDINATES)

F2_Faustini:
  Location: -87.39°S, 82.31°E
  Expected CPR: 1.95

  ✓ Found in EAST:
    Pixel: row=5540, col=9511
    Stereographic: X=78446m, Y=10592m
    Pixels: 20,044

    CPR:
      Mean:   0.5901
      Median: 0.5460
      Max:    1.8572
      Expected: 1.9500
      ⚠ 69.7% error from expected

    SRD: mean=0.6326
    Ice (CPR>1.0, SRD<0.2): 19 pixels
    Ice (CPR>0.8, SRD<0.3): 96 pixels

H3_Haworth:
  Location: -86.09°S, 97.5°E
  Expected CPR: 1.57

  ✓ Found in EAST:
    Pixel: row=6583, col=11077
    Stereographic: X=117596m, Y=-15482m
    Pixels: 19,252

    CPR:
      Mean:   0.3962
      Median: 0.3632
      Max:    1.5191
      Expected: 1.5700
      ⚠ 74.8% error from expected

    SRD: mean=0.7229
    Ice (CPR>1.0, SRD<0.2): 1 pixels
    Ice (CPR>0.8, SRD<0.3): 11 pixels

H1_Haworth:
  Location: -86.09°S, 98.0°E
  Expected CPR: 0.80

  ✓ Found in EAST:
    Pixel: 

# FINAL CODE

**1.Ice Detection Method**
What we use:
CPR (Circular Polarization Ratio) = how much radar bounces back in same-circular vs opposite-circular polarization
SRD (depolarization) = how much the radar wave gets scrambled after hitting the surface

Why both?
Ice gives CPR > 1.0 (bounces strongly in circular mode)
Ice gives SRD < 0.2 (scrambles the wave = depolarizes it)
Rocks also give high CPR BUT SRD > 0.5 (don't depolarize)

Threshold:
High confidence ice = CPR > 1.0 AND SRD < 0.2
Medium confidence ice = CPR > 0.8 AND SRD < 0.3
This filters out false positives (rocks).

**2.Volume Calculation Formula**

# Ice Volume = (Number of ice pixels) × (pixel area) × (radar depth) × (ice concentration)

Parameters:
Pixel area = 25m × 25m = 625 m²
Radar depth = 5m (S-band radar penetrates ~5m into regolith)
Ice concentration = 2% to 8% by volume (varies)

Example:
6,601 high-conf ice pixels
Area = 6,601 × 625 = 4.1M m²
Volume = 4.1M m² × 5m = 20.5M m³ regolith
Ice (5% concentration) = 20.5M × 0.05 = 1.0M m³ ice

Uncertainty range:
Conservative (2%): 0.4M m³
Best (5%): 1.0M m³
Optimistic (8%): 1.6M m³


**3.Your Results Summary**

MetricValueHigh confidence ice pixels6,601Medium confidence ice pixels43,854Best ice volume estimate4.5 million m³Range1.78 — 7.13 million m³

**4.What the Graphs Mean**

Graph 1: CPR Histogram (left)
X-axis = CPR value (0 to 5)
Y-axis = number of pixels (log scale = can see rare ice pixels)
Big peak at 0.2 = background (non-ice rocks, regolith)
Red line at CPR=1.0 = ice threshold
Tiny peak beyond 1.0 = ice candidates (sparse, as expected)

Graph 2: Cumulative Distribution (right)
Shows what % of pixels are below each CPR value
Example: 99% of pixels have CPR < 0.6 (non-ice background)
Only ~0.2% above CPR=1.0 (ice pixels are rare, spatially concentrated)

Graph 3: CPR Map (linear scale)
Visual image of CPR values across crater
Dark = low CPR (no ice)
Bright = high CPR (ice)

Graph 4: Log Scale CPR
Same data but stretched logarithmically to see both bright AND dim features
Easier to spot rare ice pixels that would be invisible in linear scale

Graph 5: Ice Probability Map
Red = no ice (CPR < 0.8)
Yellow = medium confidence ice (0.8 < CPR < 1.0)
Green = high confidence ice (CPR > 1.0)

In [4]:
from osgeo import gdal, osr
import numpy as np
import matplotlib.pyplot as plt
import os

gdal.PushErrorHandler("CPLQuietErrorHandler")

# ============================================================================
# CONFIGURATION
# ============================================================================

base_dir = r"E:\ISRO_IMG"
output_dir = r"E:\output\ice_analysis"
os.makedirs(output_dir, exist_ok=True)

files = {
    "east": {
        "cpr": r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspeast_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspeast_d_cpr_xx_fp_xx_xxx.tif",
        "srd": r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspeast_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspeast_d_srd_xx_fp_xx_xxx.tif",
    },
    "west": {
        "cpr": r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspwest_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspwest_d_cpr_xx_fp_xx_xxx.tif",
        "srd": r"E:\ISRO_IMG\ch2_sar_ndxl_20250630mpcpspwest_d_fp_xxx\data\derived\20250630\ch2_sar_ndxl_20250630mpcpspwest_d_srd_xx_fp_xx_xxx.tif",
    },
}

craters = {
    "F2_Faustini": {"lat": -87.39, "lon": 82.31, "expected_cpr": 1.95, "radius_m": 2000},
    "H3_Haworth": {"lat": -86.09, "lon": 97.50, "expected_cpr": 1.57, "radius_m": 2000},
    "H1_Haworth": {"lat": -86.09, "lon": 98.00, "expected_cpr": 0.80, "radius_m": 2000},
    "S1_Shoemaker": {"lat": -89.33, "lon": 76.92, "expected_cpr": 1.40, "radius_m": 2000},
}

# ============================================================================
# LOAD DATA
# ============================================================================

print("=" * 70)
print("LOADING DATA")
print("=" * 70)

swath_data = {}
for swath_name, swath_files in files.items():
    cpr_ds = gdal.Open(swath_files["cpr"])
    srd_ds = gdal.Open(swath_files["srd"])

    cpr = cpr_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)
    srd = srd_ds.GetRasterBand(1).ReadAsArray().astype(np.float32)

    swath_data[swath_name] = {
        "cpr": cpr,
        "srd": srd,
        "geotransform": cpr_ds.GetGeoTransform(),
        "projection_wkt": cpr_ds.GetProjection(),
        "shape": (cpr_ds.RasterYSize, cpr_ds.RasterXSize),
        "ds": cpr_ds,
    }
    print(f"\n{swath_name.upper()}:")
    print(f"  Shape: {swath_data[swath_name]['shape']}")
    print(f"  CPR range: {np.nanmin(cpr):.4f} -> {np.nanmax(cpr):.4f}")
    print(f"  SRD range: {np.nanmin(srd):.4f} -> {np.nanmax(srd):.4f}")

# ============================================================================
# COMPUTE ICE PROBABILITY
# ============================================================================

print("\n" + "=" * 70)
print("COMPUTING ICE PROBABILITY")
print("=" * 70)

ice_maps = {}
all_ice_prob = []
summary_results = []

for swath_name, swath in swath_data.items():
    cpr = swath["cpr"]
    srd = swath["srd"]

    ice_probability = np.zeros_like(cpr)

    high_conf = (cpr > 1.0) & (srd < 0.2)
    ice_probability[high_conf] = 1.0

    med_conf = (cpr > 0.8) & (srd < 0.3) & ~high_conf
    ice_probability[med_conf] = 0.5

    ice_maps[swath_name] = ice_probability
    all_ice_prob.append(ice_probability[~np.isnan(ice_probability)])

    ice_high = high_conf.sum()
    ice_med = med_conf.sum()

    print(f"\n{swath_name.upper()}:")
    print(f"  High conf ice pixels (CPR>1.0, SRD<0.2): {ice_high:,}")
    print(f"  Med conf ice pixels (CPR>0.8, SRD<0.3): {ice_med:,}")

    pixel_size_m = 25
    pixel_area_m2 = pixel_size_m ** 2
    depth_m = 5.0

    ice_pixels_eff = ice_high + 0.5 * ice_med
    regolith_volume_m3 = ice_pixels_eff * pixel_area_m2 * depth_m

    ice_vol_min = regolith_volume_m3 * 0.02
    ice_vol_best = regolith_volume_m3 * 0.05
    ice_vol_max = regolith_volume_m3 * 0.08

    print("\n  Ice Volume Estimates:")
    print(f"    Conservative (2%): {ice_vol_min:.3e} m3 ({ice_vol_min / 1e6:.2f}M m3)")
    print(f"    Best (5%):         {ice_vol_best:.3e} m3 ({ice_vol_best / 1e6:.2f}M m3)")
    print(f"    Optimistic (8%):   {ice_vol_max:.3e} m3 ({ice_vol_max / 1e6:.2f}M m3)")

    summary_results.append(
        {
            "swath": swath_name,
            "high_conf": ice_high,
            "med_conf": ice_med,
            "vol_min": ice_vol_min,
            "vol_best": ice_vol_best,
            "vol_max": ice_vol_max,
        }
    )

    output_name = f"ice_probability_{swath_name}.tif"
    output_path = os.path.join(output_dir, output_name)

    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(output_path, cpr.shape[1], cpr.shape[0], 1, gdal.GDT_Float32)
    out_ds.SetGeoTransform(swath["geotransform"])
    out_ds.SetProjection(swath["projection_wkt"])
    out_ds.GetRasterBand(1).WriteArray(ice_probability)
    out_ds.FlushCache()
    del out_ds

    print(f"  Saved: {output_name}")

# ============================================================================
# GLOBAL SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("GLOBAL SOUTH POLE SUMMARY")
print("=" * 70)

all_ice = np.concatenate(all_ice_prob)
total_high = sum(r["high_conf"] for r in summary_results)
total_med = sum(r["med_conf"] for r in summary_results)
total_vol_min = sum(r["vol_min"] for r in summary_results)
total_vol_best = sum(r["vol_best"] for r in summary_results)
total_vol_max = sum(r["vol_max"] for r in summary_results)

print("\nIce Detection Summary:")
print(f"  Total high conf pixels: {total_high:,}")
print(f"  Total med conf pixels: {total_med:,}")

print("\nGlobal South Pole Ice Volume:")
print(f"  Range: [{total_vol_min:.3e}, {total_vol_max:.3e}] m3")
print(f"  Best estimate: {total_vol_best:.3e} m3")
print(f"  In millions: {total_vol_best / 1e6:.1f} million m3")

# ============================================================================
# PER-CRATER EXTRACTION
# ============================================================================

print("\n" + "=" * 70)
print("PER-CRATER ICE DETECTION")
print("=" * 70)


def latlon_to_pixel_correct(lat, lon, ds, radius_m=2000):
    """Convert Moon lat/lon to pixel coordinates."""
    source_srs = osr.SpatialReference()
    source_srs.ImportFromWkt(
        """GEOGCS["GCS_Moon_2000",
        DATUM["D_Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],
        PRIMEM["Reference_Meridian",0],
        UNIT["degree",0.0174532925199433]]"""
    )

    target_srs = osr.SpatialReference()
    target_srs.ImportFromWkt(ds.GetProjection())

    transform = osr.CoordinateTransformation(source_srs, target_srs)
    x, y, z = transform.TransformPoint(lon, lat)

    gt = ds.GetGeoTransform()
    pixel_col = (x - gt[0]) / gt[1]
    pixel_row = (y - gt[3]) / gt[5]

    return int(pixel_row), int(pixel_col), x, y


crater_results = []

for crater_name, crater_info in craters.items():
    lat = crater_info["lat"]
    lon = crater_info["lon"]
    expected_cpr = crater_info["expected_cpr"]
    radius_m = crater_info["radius_m"]

    print(f"\n{crater_name}:")
    print(f"  Location: {lat} deg S, {lon} deg E")
    print(f"  Expected CPR: {expected_cpr:.2f}")

    found = False
    for swath_name, swath in swath_data.items():
        try:
            row, col, x, y = latlon_to_pixel_correct(lat, lon, swath["ds"])

            if 0 <= row < swath["shape"][0] and 0 <= col < swath["shape"][1]:
                radius_pixels = radius_m / 25.0

                yy, xx = np.ogrid[: swath["shape"][0], : swath["shape"][1]]
                mask = (xx - col) ** 2 + (yy - row) ** 2 <= radius_pixels ** 2

                cpr_region = swath["cpr"][mask]
                srd_region = swath["srd"][mask]

                # Create combined valid mask so CPR and SRD remain aligned.
                valid_mask = ~np.isnan(cpr_region) & ~np.isnan(srd_region)
                cpr_valid = cpr_region[valid_mask]
                srd_valid = srd_region[valid_mask]

                if len(cpr_valid) > 100:
                    cpr_mean = np.mean(cpr_valid)
                    cpr_max = np.max(cpr_valid)
                    cpr_median = np.median(cpr_valid)
                    srd_mean = np.mean(srd_valid)

                    print(f"\n  Found in {swath_name.upper()}:")
                    print(f"    Pixel: row={row}, col={col}")
                    print(f"    Stereographic: X={x:.0f}m, Y={y:.0f}m")
                    print(f"    Pixels: {len(cpr_valid):,}")

                    print("\n    CPR Statistics:")
                    print(f"      Mean:   {cpr_mean:.4f}")
                    print(f"      Median: {cpr_median:.4f}")
                    print(f"      Max:    {cpr_max:.4f}")
                    print(f"      Expected: {expected_cpr:.4f}")

                    error_pct = abs(cpr_mean - expected_cpr) / expected_cpr * 100
                    if error_pct < 30:
                        print(f"      MATCH ({error_pct:.1f}% error)")
                    else:
                        print(f"      WARNING: {error_pct:.1f}% error")

                    print(f"\n    SRD: mean={srd_mean:.4f}")

                    ice_high = ((cpr_valid > 1.0) & (srd_valid < 0.2)).sum()
                    ice_med = ((cpr_valid > 0.8) & (srd_valid < 0.3)).sum()
                    print(f"    Ice pixels (high): {ice_high:,}")
                    print(f"    Ice pixels (med): {ice_med:,}")

                    crater_results.append(
                        {
                            "crater": crater_name,
                            "swath": swath_name,
                            "cpr_mean": cpr_mean,
                            "expected": expected_cpr,
                            "error_pct": error_pct,
                            "ice_high": ice_high,
                            "pixels": len(cpr_valid),
                        }
                    )

                    found = True
                    break
        except Exception:
            pass

    if not found:
        print("  WARNING: Not found in coverage area")

# ============================================================================
# CRATER SUMMARY TABLE
# ============================================================================

if crater_results:
    print("\n" + "=" * 70)
    print("CRATER SUMMARY TABLE")
    print("=" * 70)
    print(f"\n{'Crater':<20} {'CPR Mean':<12} {'Expected':<10} {'Error %':<10} {'Ice High':<10}")
    print("-" * 70)
    for r in crater_results:
        print(
            f"{r['crater']:<20} {r['cpr_mean']:<12.4f} "
            f"{r['expected']:<10.4f} {r['error_pct']:<10.1f} {r['ice_high']:<10,}"
        )

# ============================================================================
# VISUALIZATION - HANDLE LARGE SWATHS
# ============================================================================

print("\n" + "=" * 70)
print("GENERATING VISUALIZATIONS")
print("=" * 70)

for swath_name, swath in swath_data.items():
    cpr = swath["cpr"]
    srd = swath["srd"]
    ice_prob = ice_maps[swath_name]

    # For large WEST swath, downsample for visualization
    if swath_name == "west":
        downsample_factor = 4
        cpr_viz = cpr[::downsample_factor, ::downsample_factor]
        srd_viz = srd[::downsample_factor, ::downsample_factor]
        ice_prob_viz = ice_prob[::downsample_factor, ::downsample_factor]
        print(f"\n{swath_name.upper()}: Downsampled 4x for visualization ({cpr_viz.shape})")
    else:
        cpr_viz = cpr
        srd_viz = srd
        ice_prob_viz = ice_prob

    cpr_valid = cpr_viz[~np.isnan(cpr_viz)]

    vmin = np.percentile(cpr_valid, 2)
    vmax = np.percentile(cpr_valid, 98)

    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    fig.suptitle(f"Ice Detection Analysis - {swath_name.upper()} Swath", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    im1 = ax.imshow(cpr_viz, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"CPR (Linear Scale, {vmin:.3f}-{vmax:.3f})", fontsize=12)
    plt.colorbar(im1, ax=ax, label="CPR")

    ax = axes[0, 1]
    cpr_log = np.log10(np.maximum(cpr_viz, 1e-3))
    im2 = ax.imshow(cpr_log, cmap="plasma")
    ax.set_title("CPR (Log10 Scale)", fontsize=12)
    plt.colorbar(im2, ax=ax, label="log10(CPR)")

    ax = axes[1, 0]
    ice_display = ice_prob_viz.copy()
    ice_display[np.isnan(cpr_viz)] = np.nan
    im3 = ax.imshow(ice_display, cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_title("Ice Probability Map (0=no ice, 0.5=medium, 1.0=high)", fontsize=12)
    plt.colorbar(im3, ax=ax, label="Ice Probability")

    ax = axes[1, 1]
    cpr_hist = cpr[~np.isnan(cpr)]
    ax.hist(cpr_hist, bins=200, color="steelblue", alpha=0.7, edgecolor="black")
    ax.axvline(1.0, color="red", linestyle="--", linewidth=2, label="Ice threshold (CPR=1.0)")
    ax.axvline(
        np.median(cpr_hist),
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Median ({np.median(cpr_hist):.3f})",
    )
    ax.set_xlabel("CPR", fontsize=11)
    ax.set_ylabel("Frequency", fontsize=11)
    ax.set_title("CPR Distribution", fontsize=12)
    ax.set_yscale("log")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    output_path = os.path.join(output_dir, f"ice_analysis_{swath_name}.png")
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"Saved: ice_analysis_{swath_name}.png")
    plt.close()

# ============================================================================
# COMBINED STATISTICS PLOT
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("South Pole Ice Detection - Combined Statistics", fontsize=14, fontweight="bold")

ax = axes[0]
for swath_name, swath in swath_data.items():
    cpr = swath["cpr"]
    cpr_valid = cpr[~np.isnan(cpr)]
    ax.hist(cpr_valid, bins=200, alpha=0.6, label=swath_name.upper(), edgecolor="black")

ax.axvline(1.0, color="red", linestyle="--", linewidth=2.5, label="Ice threshold")
ax.set_xlabel("CPR", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("CPR Distribution (Both Swaths)", fontsize=12)
ax.set_yscale("log")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

ax = axes[1]
for swath_name, swath in swath_data.items():
    cpr = swath["cpr"]
    cpr_valid = np.sort(cpr[~np.isnan(cpr)])
    cumulative = np.arange(1, len(cpr_valid) + 1) / len(cpr_valid)
    ax.plot(cpr_valid, cumulative, linewidth=2, label=swath_name.upper())

ax.axvline(1.0, color="red", linestyle="--", linewidth=2.5, label="Ice threshold")
ax.set_xlabel("CPR", fontsize=12)
ax.set_ylabel("Cumulative Fraction", fontsize=12)
ax.set_title("Cumulative CPR Distribution", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
output_path = os.path.join(output_dir, "combined_statistics.png")
plt.savefig(output_path, dpi=150, bbox_inches="tight")
print("Saved: combined_statistics.png")
plt.close()

# ============================================================================
# FINAL REPORT
# ============================================================================

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print(f"\nAll outputs saved to: {output_dir}")
print("\nGenerated files:")
print("  - ice_probability_east.tif")
print("  - ice_probability_west.tif")
print("  - ice_analysis_east.png")
print("  - ice_analysis_west.png")
print("  - combined_statistics.png")
print("\nKey Results:")
print(f"  Global South Pole Ice Volume: {total_vol_best / 1e6:.1f} million m3")
print(f"  Uncertainty Range: [{total_vol_min / 1e6:.2f}, {total_vol_max / 1e6:.2f}] million m3")
print(f"  High Confidence Ice Pixels: {total_high:,}")
print(f"  Medium Confidence Ice Pixels: {total_med:,}")

print("\n" + "=" * 70)


LOADING DATA

EAST:
  Shape: (12237, 12794)
  CPR range: 0.0030 -> 4.0782
  SRD range: 0.0018 -> 0.9996

WEST:
  Shape: (24794, 24181)
  CPR range: 0.0009 -> 4.9193
  SRD range: 0.0006 -> 20.9513

COMPUTING ICE PROBABILITY

EAST:
  High conf ice pixels (CPR>1.0, SRD<0.2): 2,787
  Med conf ice pixels (CPR>0.8, SRD<0.3): 13,658

  Ice Volume Estimates:
    Conservative (2%): 6.010e+05 m3 (0.60M m3)
    Best (5%):         1.502e+06 m3 (1.50M m3)
    Optimistic (8%):   2.404e+06 m3 (2.40M m3)
  Saved: ice_probability_east.tif

WEST:
  High conf ice pixels (CPR>1.0, SRD<0.2): 3,814
  Med conf ice pixels (CPR>0.8, SRD<0.3): 30,196

  Ice Volume Estimates:
    Conservative (2%): 1.182e+06 m3 (1.18M m3)
    Best (5%):         2.955e+06 m3 (2.96M m3)
    Optimistic (8%):   4.728e+06 m3 (4.73M m3)
  Saved: ice_probability_west.tif

GLOBAL SOUTH POLE SUMMARY

Ice Detection Summary:
  Total high conf pixels: 6,601
  Total med conf pixels: 43,854

Global South Pole Ice Volume:
  Range: [1.783e+06, 

C:\Users\KIIT\AppData\Local\Temp\ipykernel_30352\3338815074.py:400: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\KIIT\AppData\Local\Temp\ipykernel_30352\3338815074.py:402: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(output_path, dpi=150, bbox_inches="tight")


Saved: combined_statistics.png

ANALYSIS COMPLETE

All outputs saved to: E:\output\ice_analysis

Generated files:
  - ice_probability_east.tif
  - ice_probability_west.tif
  - ice_analysis_east.png
  - ice_analysis_west.png
  - combined_statistics.png

Key Results:
  Global South Pole Ice Volume: 4.5 million m3
  Uncertainty Range: [1.78, 7.13] million m3
  High Confidence Ice Pixels: 6,601
  Medium Confidence Ice Pixels: 43,854

